# Session 12: Random Forest — Titanic Dataset


**Dataset:** Titanic, loaded via `sns.load_dataset('titanic')`, continued from the Decision Trees session
**Input:** 891 passenger records, mixed numeric and categorical predictors
**Target:** `survived` (0 = did not survive, 1 = survived)

This notebook works through 8 exercises: Random Forest theory (bagging + random feature selection), reusing the cleaned Titanic split from the Decision Trees session, out-of-bag (OOB) validation, comparing a Random Forest against a single Decision Tree, tuning `max_features`, the `n_estimators` diminishing-returns curve, and comparing feature importance rankings.


## Setup
Import libraries used throughout the notebook.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                              precision_score, recall_score, f1_score, classification_report)

%matplotlib inline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


---
## Q1. What is a Random Forest?
What is a Random Forest, and how does it build on a single decision tree? Explain the two sources of randomness that give the model its name: (1) bootstrap sampling — each tree is trained on a random sample of rows drawn with replacement - and (2) random feature selection - each split only considers a random subset of features. Why does averaging predictions over many such 'decorrelated' trees reduce overfitting compared to relying on a single, deep decision tree?


 A **Random Forest** is an ensemble of many decision trees whose individual predictions are combined (by majority vote for classification, or averaging for regression) into a single final prediction. It builds directly on the single decision tree algorithm - each tree in the forest is grown the same way a standalone Decision Tree would be - but introduces two deliberate sources of randomness so that the trees don't all learn the same thing:

1. **Bootstrap sampling ("bagging"):** each tree is trained on a random sample of the training rows, drawn *with replacement*, the same size as the original training set. Because sampling is with replacement, each tree sees a slightly different subset of passengers (some rows repeated, others left out entirely - see Q3), so no two trees are trained on identical data.
2. **Random feature selection:** at every split, instead of considering all available features (as a single Decision Tree would), each tree only considers a random subset of them (controlled by `max_features`, see Q6). This stops a single dominant feature (like `sex` in this dataset) from being chosen at the top of every tree, forcing different trees to discover different, sometimes weaker, but still useful splitting patterns.

Together these two randomizations produce trees that are individually noisier/weaker but **decorrelated** from one another - their errors tend to be different and somewhat independent rather than all making the same mistakes. Averaging many decorrelated predictors cancels out a lot of that individual noise (much like averaging many noisy measurements gets you closer to the true value), which is why a forest of shallow, imperfect trees generalizes better than a single very deep tree that has been allowed to fit the training data (including its noise) as closely as possible.

---
## Q2. Reuse the Cleaned Titanic Dataset and Train/Test Split
Reuse the cleaned Titanic dataset and train/test split from the Decision Trees session. Confirm the shape of your training and test sets, and briefly restate which features and encodings you carried over. Keeping the same split lets you compare a Random Forest directly against the single decision tree you trained previously.


In [ ]:
# Load Titanic dataset and check its shape.



# --- Same cleaning/encoding pipeline as the Decision Trees session ---

# Age → Fill missing values with median (reasonable for numeric data).
# Embarked → Fill missing values with mode (most frequent port).
# Fare → Fill missing values with median (reasonable for numeric data).




# Drop redundant/high-missing columns (deck has too many NaNs; class/who/adult_male/alive/alone
# duplicate information already captured by pclass/sex/age/sibsp/parch)




# Encode categoricals: Sex -> binary, 
# Embarked -> one-hot (drop_first to avoid the dummy trap)




# Select features and target variable for modeling




# Split the dataset into training and test sets (80% train, 20% test), stratifying by the target variable to maintain class balance.





print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nFeatures used:", features)
print("\nAny missing values left?\n", X.isnull().sum().sum(), "missing values total")


---
## Q3. Bootstrap Sampling and Out-of-Bag (OOB) Samples
When a bootstrap sample is drawn with replacement from N training rows, on average only about 63.2% of the unique rows are included in any given tree's sample. What happens to the remaining rows, and how does scikit-learn use them to compute an 'out-of-bag score' as a built-in validation estimate — without needing a separate held-out validation set?


In [ ]:
# Demonstrate the ~63.2% figure empirically with a quick simulation
N = len(X_train)
rng = np.random.default_rng(RANDOM_STATE)

bootstrap_idx = rng.integers(0, N, size=N)         # sample N indices WITH replacement
unique_included = len(set(bootstrap_idx))
pct_included = unique_included / N * 100

print(f"Training rows (N): {N}")
print(f"Unique rows included in one bootstrap sample: {unique_included}")
print(f"Percentage of rows included: {pct_included:.2f}%")
print(f"Theoretical limit as N -> infinity: {(1 - np.exp(-1)) * 100:.2f}%  (i.e. 1 - 1/e)")
print(f"Percentage of rows LEFT OUT (out-of-bag) for this tree: {100 - pct_included:.2f}%")


---
## Q4. Fit a Random Forest and Compare to the Single Decision Tree
Fit a `RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)` on the training data. Report the model's OOB score, and compare it to the test accuracy of the single `DecisionTreeClassifier` from the previous session. Did the forest improve on the single tree, and is that consistent with what Q1 predicted?


In [ ]:
# --- Single Decision Tree, as trained in the previous session (max_depth=4) ---






# --- Random Forest ---





print("\n=== Random Forest (200 trees) ===")

# print the OOB score and accuracy metrics for the Random Forest model
print(f"OOB score:      {rf.oob_score_:.4f}")
print(f"Train accuracy: {rf_train_acc:.4f}")
print(f"Test accuracy:  {rf_test_acc:.4f}")

print("\n=== Comparison ===")
print(f"Decision Tree test accuracy: {dt_test_acc:.4f}")
print(f"Random Forest OOB score:     {rf.oob_score_:.4f}")
print(f"Random Forest test accuracy: {rf_test_acc:.4f}")


---
## Q5. Predictions, Confusion Matrix, and Metric Comparison
Generate predictions on the test set, build the confusion matrix, and compute accuracy, precision, recall, and F1-score. Put these side by side with the single decision tree's metrics from the previous session in a small comparison table. Which metric improved the most, and does the gap between training accuracy and test accuracy look smaller than it did for the single tree?


In [ ]:
# Compare the performance of the Decision Tree and Random Forest models




# Confusion matrices for both models




# Visualize the confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ConfusionMatrixDisplay(cm_dt, display_labels=['died', 'survived']).plot(ax=axes[0], cmap='Oranges', colorbar=False)
axes[0].set_title("Decision Tree — Test Set")
ConfusionMatrixDisplay(cm_rf, display_labels=['died', 'survived']).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title("Random Forest — Test Set")
plt.tight_layout()
plt.show()

metrics_comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Train Accuracy': [dt_train_acc, rf_train_acc],
    'Test Accuracy': [accuracy_score(y_test, dt_pred), accuracy_score(y_test, rf_pred)],
    'Precision': [precision_score(y_test, dt_pred), precision_score(y_test, rf_pred)],
    'Recall': [recall_score(y_test, dt_pred), recall_score(y_test, rf_pred)],
    'F1-score': [f1_score(y_test, dt_pred), f1_score(y_test, rf_pred)],
    'Train-Test Gap': [dt_train_acc - accuracy_score(y_test, dt_pred),
                        rf_train_acc - accuracy_score(y_test, rf_pred)]
})
metrics_comparison


---
## Q6. The `max_features` Parameter
Explain what `max_features` controls (the number of features randomly considered at each split) and why restricting it — rather than letting every split see all features — helps decorrelate the trees in the forest. Using 5-fold cross-validation, compare model accuracy for `max_features='sqrt'`, `max_features='log2'`, and `max_features=None`. What trade-off do you observe?


In [ ]:
max_features_options = ['sqrt', 'log2', None]
results = []

for mf in max_features_options:
    rf_mf = RandomForestClassifier(n_estimators=200, max_features=mf, random_state=RANDOM_STATE)
    scores = cross_val_score(rf_mf, X_train, y_train, cv=5, scoring='accuracy')
    results.append({
        'max_features': str(mf),
        'mean_cv_accuracy': scores.mean(),
        'std_cv_accuracy': scores.std()
    })
    print(f"max_features={mf}: mean CV accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

results_df = pd.DataFrame(results)

plt.figure(figsize=(6, 4))
plt.bar(results_df['max_features'], results_df['mean_cv_accuracy'],
        yerr=results_df['std_cv_accuracy'], color=['steelblue', 'seagreen', 'indianred'], capsize=5)
plt.ylabel("Mean 5-fold CV Accuracy")
plt.title("Random Forest Accuracy by max_features")
plt.ylim(0.7, 0.85)
plt.show()


---
## Q7. How Many Trees Are Enough?
Train Random Forests with `n_estimators` ranging over `[10, 25, 50, 100, 200, 400]`, recording OOB score (or test accuracy) at each value, and plot the result. At what point do returns diminish? Unlike increasing `max_depth` in a single tree, why doesn't adding more trees cause the forest to overfit?


In [ ]:
n_estimators_range = [10, 25, 50, 100, 200, 400]
oob_scores = []
test_accs = []

for n in n_estimators_range:
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=RANDOM_STATE)
    rf_n.fit(X_train, y_train)
    oob_scores.append(rf_n.oob_score_)
    test_accs.append(accuracy_score(y_test, rf_n.predict(X_test)))
    print(f"n_estimators={n:>3}: OOB score = {rf_n.oob_score_:.4f}, test accuracy = {test_accs[-1]:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(n_estimators_range, oob_scores, marker='o', label='OOB score', color='steelblue')
plt.plot(n_estimators_range, test_accs, marker='s', label='Test accuracy', color='seagreen')
plt.xlabel("n_estimators (number of trees)")
plt.ylabel("Score")
plt.title("Random Forest Performance vs. Number of Trees")
plt.legend()
plt.show()


---
## Q8. Feature Importance
Extract and plot `feature_importances_` from your Random Forest, ranked from most to least important, and place it next to the single decision tree's feature importance ranking from the previous session. Are the rankings similar? Why do feature importances from a Random Forest tend to be more stable and trustworthy than those from a single tree, given how each is computed?


In [ ]:
dt_importances = pd.Series(dt.feature_importances_, index=features).sort_values(ascending=False)
rf_importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

dt_importances.plot(kind='barh', ax=axes[0], color='darkorange')
axes[0].set_title("Decision Tree — Feature Importances")
axes[0].invert_yaxis()
axes[0].set_xlabel("Importance")

rf_importances.plot(kind='barh', ax=axes[1], color='seagreen')
axes[1].set_title("Random Forest — Feature Importances")
axes[1].invert_yaxis()
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

comparison_importance = pd.DataFrame({
    'Decision Tree': dt_importances,
    'Random Forest': rf_importances
}).round(3)
comparison_importance


## Q9

##  Train a Randomforest classifier with following settings and compare the results:

1. Case 1:
- n_estimators=200
- bootstrap=True
- max_samples=None
- oob_score=True
- max_features="sqrt"
- random state=42

2. Case 2:
- n_estimators=200
- bootstrap=True
- max_samples=0.5 (50% of Training set Size)
- oob_score=True
- max_features="sqrt"
- random state=42
